# Q6. Conditional transfer entropy of population spikes

Run `q6_cte.py` followed by `q6_cte_stats.R`, then run this notebook. It reads and saves plots in `results/q6/conditional_transfer_entropy` by default. Set `Q6_CTE_OUTPUT_DIR` to use another run.

For source A and target B, estimate the conditional mutual information between A's past and B's current spike count, given B's past, the third area's past and B's slow activity. The response categories are 0, 1, 2, 3, 4 and 5+ spikes. Each area's short history uses past counts in 0–20 ms and 20 ms to the history limit. The slow target state uses 1–5 s past counts. Tertile thresholds and smoothed count tables are estimated in training time blocks, and the log-probability gain is scored in a held-out block. Negative estimates can occur when the full table predicts worse than the reduced table.

Bits per original target spike is the primary comparison metric, with bits per bin also reported. Each mouse-level point averages the matched-unit draws for that mouse. The results are specific to these binned states and represent predictive dependence. [Transfer entropy from neural spikes](https://pmc.ncbi.nlm.nih.gov/articles/PMC8084348/)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if 'Q6_CTE_OUTPUT_DIR' in os.environ:
    output = Path(os.environ['Q6_CTE_OUTPUT_DIR'])
else:
    repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / 'code/questions/q6_cte.py').is_file())
    output = repo / 'results/q6/conditional_transfer_entropy'
mice = pd.read_csv(output / 'q6_cte_mouse_scores.csv')
means = pd.read_csv(output / 'q6_cte_stats_means.csv')
tests = pd.read_csv(output / 'q6_cte_stats_primary_tests.csv')
mice['direction'] = mice['source'].astype(str) + '->' + mice['target'].astype(str)
candidates = ['1->2', '3->2', '3->1']
reverse = ['2->1', '2->3', '1->3']
colors = dict(zip(candidates, ['#1B6CA8', '#A24A12', '#6B4B91']))

def label(direction):
    return direction.replace('->', ' → ')

primary = mice.loc[mice['primary']]
extra = primary.groupby('direction')[['bits_per_bin', 'seen_context_fraction']].mean()
primary_summary = means.loc[means['primary']].merge(extra, on='direction')
# Descriptive only. The local shift also changes the relationship with the conditioning variables.
primary_summary['aligned_minus_shifted_bits_per_spike'] = primary_summary['estimate'] - primary_summary['shifted_mean']
display(primary_summary)
display(tests)

In [ ]:
values = primary.pivot(index='mouse_id', columns='direction', values='bits_per_spike')
summary = means.loc[means['primary']].set_index('direction')
fig, axes = plt.subplots(1, 2, figsize=(9, 4.3), sharey=True, layout='constrained')
for ax, directions, title in zip(axes, [candidates, reverse], ['Q6 candidates', 'Reverse directions']):
    x = np.arange(len(directions))
    ax.plot(x, values[directions].to_numpy().T, 'o-', color='0.55', ms=3, lw=0.75, alpha=0.4)
    for i, direction in enumerate(directions):
        row = summary.loc[direction]
        ax.errorbar(i + 0.11, row.estimate,
                    yerr=[[row.estimate - row.ci95_low], [row.ci95_high - row.estimate]],
                    fmt='o', lw=2, capsize=4, color=colors.get(direction, '#343434'))
    ax.axhline(0, color='0.6', linestyle='--', lw=0.8)
    ax.set_xticks(x, [label(d) for d in directions])
    ax.set(xlim=(-0.25, 2.35), xlabel='Source → target', title=title)
axes[0].set_ylabel('Conditional transfer entropy (bits per target spike)')
fig.suptitle('Conditional transfer entropy · 10 ms bins, 100 ms history')
fig.savefig(output / 'q6_cte_primary.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
candidate_means = means.loc[means['candidate']]
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True, layout='constrained')
panels = [(candidate_means.query('history_ms == 100'), 'bin_ms', 10, 'Bin width (ms)', '100 ms history'),
          (candidate_means.query('bin_ms == 10'), 'history_ms', 100, 'History window (ms)', '10 ms bins')]
for ax, (data, column, primary_value, xlabel, title) in zip(axes, panels):
    settings = sorted(data[column].unique())
    primary_idx = settings.index(primary_value)
    ax.axvspan(primary_idx - 0.24, primary_idx + 0.24, color='0.93', zorder=0)
    ax.axhline(0, color='0.6', linestyle='--', lw=0.8)
    for i, direction in enumerate(candidates):
        rows = data.loc[data['direction'] == direction].sort_values(column)
        x = np.array([settings.index(value) for value in rows[column]]) + (i - 1) * 0.14
        ax.errorbar(x, rows.estimate, yerr=[rows.estimate - rows.ci95_low, rows.ci95_high - rows.estimate],
                    fmt='o-', capsize=3, color=colors[direction], label=label(direction))
    ax.set_xticks(range(len(settings)), settings)
    ax.set(xlabel=xlabel, title=title)
axes[0].set_ylabel('Mean conditional TE (bits per target spike)')
axes[1].legend(frameon=False, title='Source → target', loc='upper left', bbox_to_anchor=(1.01, 1))
fig.suptitle('Sensitivity to bin width and history window')
fig.savefig(output / 'q6_cte_sensitivity.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
diagnostic = primary.loc[primary['candidate']].dropna(subset=['shifted_bits_per_spike'])
if not diagnostic.empty:
    fig, ax = plt.subplots(figsize=(5.4, 4.6), layout='constrained')
    for direction in candidates:
        rows = diagnostic.loc[diagnostic['direction'] == direction]
        ax.scatter(rows['shifted_bits_per_spike'], rows['bits_per_spike'],
                   s=30, alpha=0.75, color=colors[direction], label=label(direction))
    scores = diagnostic[['shifted_bits_per_spike', 'bits_per_spike']].to_numpy()
    endpoints = [min(scores.min(), 0), max(scores.max(), 0)]
    ax.plot(endpoints, endpoints, ls='--', lw=1, color='0.5', zorder=0)
    ax.set(xlabel='Shifted source (bits per target spike)', ylabel='Aligned source (bits per target spike)',
           title='Local circular-shift diagnostic', aspect='equal')
    ax.legend(frameon=False, title='Source → target')
    fig.savefig(output / 'q6_cte_shift_diagnostic.png', dpi=180, bbox_inches='tight')
    plt.show()